In [1]:
# 路径引导：无论从项目根还是 notebooks/ 目录启动内核，都能找到 src 包
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('项目根：', ROOT)

项目根： D:\agent项目集\端到端电商智能决策平台


# 03 模型训练与评估

阶段 3 的模型产物解读：流失预测、客户分群、推荐、时间序列预测。
模型由 `python -m src.pipeline churn segmentation recommender forecast modelcard` 训练，
本 notebook 只读取 `models/artifacts/` 下的结果做分析，不重训。
完整方法与局限见 `reports/model_card.md`。

In [2]:
import json
import pandas as pd
import plotly.express as px
from src import config

ART = config.MODEL_DIR
churn_metrics = json.loads((ART / 'churn_metrics.json').read_text(encoding='utf-8'))

## 1. 流失（30 天复购概率）模型比较

In [3]:
comparison = pd.read_csv(ART / 'churn_model_comparison.csv')
comparison

,model,base_rate,pr_auc,roc_auc,brier,log_loss,top10_precision,top10_recall,lift_at_10pct
0,LogisticRegression,0.003805,0.017023,0.645840,0.003788,0.024235,0.011100,0.291777,2.917566
1,RandomForest,0.003805,0.012868,0.633732,0.003781,0.024447,0.009485,0.249337,2.493193
2,LightGBM,0.003805,0.010314,0.595959,0.003792,0.025222,0.007871,0.206897,2.068819
3,Recency 规则,0.003805,0.005763,0.596395,0.009638,0.059761,0.007064,0.185676,1.856633
4,随机基线（先验概率）,0.003805,0.004046,0.503121,0.003790,0.024995,0.004339,0.114058,1.140503


> 注意：约 97% 客户为单次购买，正例（30 天内复购）占比约 1%，
> 因此用 PR-AUC / lift 而非准确率评估。ROC-AUC 在极端类别不平衡下会过于乐观。

## 2. 特征重要性

In [4]:
imp = pd.read_csv(ART / 'churn_feature_importance.csv').head(15)
fig = px.bar(imp, x='importance', y='feature', orientation='h',
             title='流失模型特征重要性 Top 15')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. 校准（预测概率 vs 实际复购率）

In [5]:
calib = pd.read_csv(ART / 'churn_calibration.csv')
fig = px.line(calib, x='mean_predicted', y='observed_rate', markers=True,
              title='概率校准（ECE=%.4f）' % calib['ece'].iloc[0])
upper = calib['mean_predicted'].max()
fig.add_scatter(x=[0, upper], y=[0, upper], mode='lines',
                name='完美校准', line=dict(dash='dash'))
fig.show()

## 4. 客户分群画像

In [6]:
profiles = pd.read_csv(ART / 'segmentation_profiles.csv')
profiles

,cluster,customers,avg_recency_days,median_recency_days,avg_frequency,share_repeat,avg_monetary,median_monetary,revenue,customer_share,revenue_share,cluster_name
0,0,22145,413.676775,394.013079,1.00000,0.0,266.133212,181.55,5893519.98,0.233130,0.374450,沉睡高价值单次客
1,1,23162,160.571957,163.093044,1.00000,0.0,258.946354,177.02,5997715.46,0.243836,0.381070,近期高价值单次客
2,2,22808,161.480491,164.289653,1.00000,0.0,62.291830,61.75,1420752.05,0.240109,0.090269,近期低价值单次客
3,3,2888,270.485605,249.728958,2.11392,1.0,308.356690,225.53,890534.12,0.030403,0.056581,复购沉睡
4,4,23987,415.043698,399.171736,1.00000,0.0,64.060341,63.70,1536615.40,0.252521,0.097630,沉睡低价值单次客


## 5. 推荐系统评估（时间切分，防泄漏）

In [7]:
rec_metrics = json.loads((ART / 'recommendation_metrics.json').read_text(encoding='utf-8'))
rows = []
for model, ks in rec_metrics['evaluation']['metrics'].items():
    for k, m in ks.items():
        rows.append({'model': model, 'k': k, **m})
pd.DataFrame(rows)

,model,k,recall,ndcg,coverage
0,popularity,k=5,0.011797,0.007552,0.002333
1,popularity,k=10,0.017241,0.009376,0.004333
2,popularity,k=20,0.025408,0.011564,0.006667
3,item_item,k=5,0.014822,0.009363,0.101333
4,item_item,k=10,0.018451,0.010582,0.118000
5,item_item,k=20,0.023896,0.011993,0.125000


## 6. 未来 30 天预测（含 80% 区间）

In [8]:
hist = pd.read_csv(ART / 'forecast_history.csv', parse_dates=['date'])
fc = pd.read_csv(ART / 'forecast_orders.csv', parse_dates=['date'])
fig = px.line(title='订单量：历史 vs 未来 30 天预测')
fig.add_scatter(x=hist['date'], y=hist['orders'], name='历史')
fig.add_scatter(x=fc['date'], y=fc['forecast'], name='预测')
fig.add_scatter(x=fc['date'], y=fc['upper_80'], name='80% 上界',
                line=dict(dash='dash', color='gray'))
fig.add_scatter(x=fc['date'], y=fc['lower_80'], name='80% 下界',
                line=dict(dash='dash', color='gray'))
fig.show()